# 07 · 단안 깊이(Depth Anything V2) + ArUco 앵커링 → 물체 높이·부피

**웹캠 사진 한 장으로 RGB-D 없이 물체별 3D(위치·바닥치수·높이·대략 부피)를 추정** 하는 실험.

**원리:**
1. Depth Anything V2 = 상대 깊이 추정(절대 미터 아님).
2. **ArUco로 보드평면의 진짜 미터 깊이를 알므로**, 보드 픽셀에서 `1/Z = A·pred + B` 를 피팅해 상대→미터로 **앵커링(보정)**.
3. 각 픽셀 3D 복원 → **보드평면 위 높이맵**. 높이>임계 영역 = 물체(평면은 높이≈0이라 자동 제외).

**보너스:** 높이 임계만으로 물체가 보드에서 깔끔히 분리된다(색·참조프레임·FastSAM 선별 불필요).

> 솔직한 한계: DA는 '추정'이라 **높이가 과소평가**되기 쉽고 가려진 뒷면은 모름 → 근사 바운딩박스/부피. 정밀은 RGB-D/다중시점 필요. 앵커링 품질은 R²로 확인.

In [ ]:
import os, sys, glob
import cv2, numpy as np
import matplotlib.pyplot as plt
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(ROOT, 'src'))
import aruco_utils as au, depth_volume as dv
print('OpenCV', cv2.__version__)

OUTPUT_DIR = os.path.join(ROOT, 'output')
SCENE_DIR = os.path.join(ROOT, 'data', 'scene_images')
SQ = 0.038
board = cv2.aruco.CharucoBoard((5, 7), SQ, SQ*22/30,
                               cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_5X5_1000))
K, dist = au.load_intrinsics(os.path.join(OUTPUT_DIR, 'camera_intrinsics.npz'))

# 첫 실행 시 모델(~100MB) 자동 다운로드. GPU(device=0) 사용.
pipe = dv.load_depth_model('depth-anything/Depth-Anything-V2-Small-hf', device=0)
print('Depth model ready')

In [ ]:
# 대상 이미지 (scene_images 또는 snap_raw_*)
cands = sorted(glob.glob(os.path.join(OUTPUT_DIR, 'snap_raw_*.png'))) or \
        sorted(glob.glob(os.path.join(SCENE_DIR, '*.*')))
img = cv2.imread(cands[-1]); print('대상:', os.path.basename(cands[-1]))

hm = dv.height_map_from_depth(img, pipe, board, K, dist, square_len=SQ)
assert hm is not None, '보드 미검출'
objs = dv.detect_objects_by_height(hm, K, dist, min_height_mm=6, min_area_px=1500)
print(f'앵커링 R^2 = {hm["r2"]:.3f}   검출 물체: {len(objs)}')
print(f'{"id":>3} | {"Hmax(mm)":>9} {"Hmean(mm)":>9} | {"footprint(mm)":>14} | {"bboxVol(cm^3)":>12}')
for i, o in enumerate(objs):
    fw, fl = o['footprint_mm']
    print(f'{i:>3} | {o["height_max_mm"]:>9.1f} {o["height_mean_mm"]:>9.1f} | {fw:>6.0f} x {fl:>4.0f}   | {o["bbox_vol_cm3"]:>12.1f}')

In [ ]:
# 시각화: 높이맵 + 검출 오버레이
fig, ax = plt.subplots(1, 2, figsize=(16, 6))
ax[0].imshow(cv2.cvtColor(dv.colorize_height(hm, 60), cv2.COLOR_BGR2RGB))
ax[0].set_title('height above board (JET, 0~60mm)'); ax[0].axis('off')
ax[1].imshow(cv2.cvtColor(dv.draw_depth_objects(hm, objs), cv2.COLOR_BGR2RGB))
ax[1].set_title('detected objects (H, footprint)'); ax[1].axis('off')
plt.tight_layout(); plt.show()

## 관찰 & 다음

- **높이 과소평가**: DA가 꼭대기를 뭉개고, 임계(6mm)가 바닥을 자름 → 실제보다 낮게. 필요시 `min_height_mm`↓, 또는 물체 상단 최댓값 보정.
- **앵커링 R²가 낮으면**(<0.9): 보드가 프레임에 더 크게/평평하게 보이도록, charuco 코너가 더 많이 잡히도록.
- **정밀도**: DA는 추정이라 절대 신뢰 금물. '어디에·대략 얼마'까진 유효. 정밀은 RGB-D/다중시점.
- **정밀 경계가 필요하면**: 이 높이-검출 마스크에 FastSAM(06) 마스크를 교차해 물체 경계를 다듬을 수 있음.
- 다음: 다중시점(ArUco 정합)으로 가려진 면까지 채워 부피 근사 개선, 실시간 연동.